# LASA-RxGuard — training on a free Colab T4

Handwritten prescription OCR (CRNN vs TrOCR) + look-alike/sound-alike drug-confusion risk flagging.

**Before you run anything:** `Runtime -> Change runtime type -> T4 GPU`.

Budget: each training run is capped at roughly **2 hours**. The two biggest ways to lose
a session are (a) forgetting to persist checkpoints and having the VM recycle at 90
minutes, and (b) discovering at minute 100 that the epoch time never fit. Cell 2 mounts
Drive for (a); every training cell prints a projection after epoch 0 for (b).

## 1. GPU check

If this says CPU, stop and change the runtime type — TrOCR fine-tuning on CPU will not finish.

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.")

## 2. Persist checkpoints to Drive

Optional but strongly recommended. A recycled VM with nothing saved means starting over.

In [ ]:
USE_DRIVE = True  # set False to keep everything on the ephemeral VM disk

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST = '/content/drive/MyDrive/lasa-rx-guard'
    os.makedirs(PERSIST, exist_ok=True)
else:
    PERSIST = '/content/lasa-persist'
    os.makedirs(PERSIST, exist_ok=True)
print("checkpoints ->", PERSIST)

## 3. Get the code

Either clone your repo or upload the project as a zip.

In [ ]:
REPO_URL = "https://github.com/TushKum/lasa-rx-guard.git"

import os, pathlib
if REPO_URL:
    !git clone -q $REPO_URL /content/lasa-rx-guard
else:
    # Fallback: upload a zip of the project.
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    !mkdir -p /content/lasa-rx-guard && unzip -q -o "$name" -d /content/lasa-rx-guard
os.chdir('/content/lasa-rx-guard')
print(pathlib.Path.cwd(), sorted(p.name for p in pathlib.Path('.').iterdir())[:12])

## 4. Install

Colab already ships torch built for its CUDA runtime, so we deliberately do **not** reinstall it — doing so is a common way to break the GPU build.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps
import lasa, torch
print("lasa ok | torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 5. Build the data

Drug corpus from RxNorm + NLEM + the ISMP/FDA pair list, then 25 OFL handwriting fonts, then the synthetic dataset.

The fonts are all Google Fonts, so this reproduces the local run exactly.

In [ ]:
!python -m lasa.data.drug_names --size 2000
!python -m lasa.data.fonts

In [ ]:
import time
t0 = time.time()
!python -m lasa.data.synth --n-train 48000 --n-val 3000 --n-test 2000 --tier medium --seed 1337 --workers 2
print(f"dataset built in {(time.time()-t0)/60:.1f} min")

Sanity-check the samples before training on them. Splits must be genuinely disjoint or the generalisation numbers are meaningless.

In [ ]:
!python -m lasa.data.synth --preview
from IPython.display import Image, display
display(Image('outputs/figures/synth_preview.png'))

import csv
rows = list(csv.DictReader(open('data/synth/labels.tsv'), delimiter='\t'))
tr_w = {r['text'] for r in rows if r['split'] == 'train'}
tr_f = {r['font'] for r in rows if r['split'] == 'train'}
uw = {r['text'] for r in rows if r['split'] == 'test_unseen_word'}
uf = {r['font'] for r in rows if r['split'] == 'test_unseen_font'}
print("word overlap train/test_unseen_word:", len(tr_w & uw), "(must be 0)")
print("font overlap train/test_unseen_font:", len(tr_f & uf), "(must be 0)")

## 6. Train the CRNN

~8.4M parameters, CNN + BiLSTM + CTC, from scratch.

Expect CER to sit near 100% for the first epoch: CTC spends its early steps in an
all-blank collapse before alignment emerges. It breaks out sharply — if it is still at
100% after epoch 2, something is wrong, don't just wait.

In [ ]:
import time, shutil
t0 = time.time()
!python -m lasa.train.train_crnn \
    --epochs 15 --batch-size 128 --lr 1e-3 --preload \
    --max-minutes 100 --out outputs/checkpoints/crnn_ctc
mins = (time.time() - t0) / 60
print(f"\nCRNN wall clock: {mins:.1f} min  (budget 120)")
shutil.copytree('outputs/checkpoints/crnn_ctc', f'{PERSIST}/crnn_ctc', dirs_exist_ok=True)
print("checkpoint persisted")

## 7. Fine-tune TrOCR

`microsoft/trocr-base-handwritten`, ~334M parameters. Subsampled to 20k images because a
full 48k epoch does not fit the budget at this model size — sampling is seeded, so the
subset is identical across runs.

If you hit CUDA OOM: drop `--batch-size` to 4 and raise `--grad-accum` to 8 (same
effective batch), or switch `--model microsoft/trocr-small-handwritten`.

In [ ]:
import time, shutil
t0 = time.time()
!python -m lasa.train.train_trocr \
    --model microsoft/trocr-base-handwritten \
    --epochs 3 --batch-size 8 --grad-accum 4 --lr 5e-5 \
    --max-train 20000 --max-eval 500 \
    --max-minutes 110 --out outputs/checkpoints/trocr_ft
mins = (time.time() - t0) / 60
print(f"\nTrOCR wall clock: {mins:.1f} min  (budget 120)")
shutil.copytree('outputs/checkpoints/trocr_ft', f'{PERSIST}/trocr_ft', dirs_exist_ok=True)
print("checkpoint persisted")

## 8. Stage-1 comparison

All four test splits, and for the CRNN all three decoding modes as separate rows. Lexicon-constrained decoding is a *different task* and is never merged into the free-decoding CER.

In [ ]:
!python -m lasa.evaluation.evaluate
from IPython.display import Markdown, display
display(Markdown(open('outputs/reports/ocr_comparison.md').read()))

## 9. Stage-2 LASA evaluation

Retrieval, pair flagging precision/recall against the ISMP/FDA ground truth, a negative control, and the signal ablation.

In [ ]:
!python -m lasa.risk.evaluate_lasa --markdown
from IPython.display import Markdown, display
display(Markdown(open('outputs/reports/lasa_eval.md').read()))

## 10. Download everything

In [ ]:
import shutil
shutil.copytree('outputs', f'{PERSIST}/outputs', dirs_exist_ok=True)
!zip -qr /content/lasa_outputs.zip outputs docs
from google.colab import files
files.download('/content/lasa_outputs.zip')

---
**Reminder:** this is an unvalidated prototype trained on synthetic handwriting, not a
medical device. The domain gap to real prescriber handwriting is unmeasured and is the
single largest source of uncertainty in these numbers.